<a href="https://colab.research.google.com/github/urmilapol/urmilapolprojects/blob/master/vishnu3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
import shutil
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from google.colab import files

# 2. Upload the template image and CSV file
print("Step 1: Upload the Vishnu Template Image (e.g. vishnu.jpg / vishnu.png):")
img_upload = files.upload()
template_img_path = list(img_upload.keys())[0]

print("\nStep 2: Upload your Vishnu Sahasranamam CSV file:")
csv_upload = files.upload()
csv_filename = list(csv_upload.keys())[0]

# Read CSV
df = pd.read_csv(csv_filename)
print(f"Loaded {len(df)} rows successfully.")

# Identify CSV columns
cols = df.columns
sr_col, shlok_col, meaning_col = cols[0], cols[1], cols[2]

# 3. Setup Fonts and Directories
output_dir = "vishnu_generated_cards"
os.makedirs(output_dir, exist_ok=True)

# Select Devanagari Font
font_path = "/usr/share/fonts/truetype/fonts-deva-extra/gargi.ttf"
if not os.path.exists(font_path):
    font_path = "/usr/share/fonts/truetype/lohit-devanagari/Lohit-Devanagari.ttf"

# 4. Helper function to wrap text neatly within card margins
def wrap_devanagari_text(text, font, max_width, draw):
    lines = []
    # Ensure text is a string
    text = str(text)
    # Split sentences/clauses
    raw_sentences = text.replace(';', ';\n').split('\n')
    for sentence in raw_sentences:
        words = sentence.strip().split()
        if not words:
            continue
        current_line = []
        for word in words:
            test_line = " ".join(current_line + [word])
            try:
                # Attempt to get text bounding box
                bbox = draw.textbbox((0, 0), test_line, font=font, language='hi')
                text_width = bbox[2] - bbox[0]
            except Exception as e:
                # If textbbox fails, assume the word is too long or problematic.
                # Treat the current word as a new line to prevent crash.
                print(f"Warning: Failed to measure text '{test_line}' with error: {e}. Forcing word to new line.")
                text_width = max_width + 1 # Force it to exceed max_width

            if text_width <= max_width:
                current_line.append(word)
            else:
                if current_line:
                    lines.append(" ".join(current_line))
                current_line = [word]
        if current_line:
            lines.append(" ".join(current_line))
    return lines

# Helper to draw bold text via slight offset overlay
def draw_bold_text(draw, position, text, font, fill, language='hi'):
    x, y = position
    draw.text((x, y), text, font=font, fill=fill, language=language)
    draw.text((x + 1, y), text, font=font, fill=fill, language=language)
    draw.text((x, y + 1), text, font=font, fill=fill, language=language)

# 5. Process and Generate Cards
base_template = Image.open(template_img_path).convert("RGBA")
W, H = base_template.size

# Text area dimensions (Adjusted to start even higher to maximize space for text)
text_start_y = int(H * 0.25) # Adjusted to start even higher on the image
margin_x = int(W * 0.12)
content_width = W - (2 * margin_x)

# Font sizes relative to image dimensions (Further reduced)
title_size = int(W * 0.052)
shlok_size = int(W * 0.028) # Adjusted shlok font size further
meaning_size = int(W * 0.022) # Adjusted meaning font size further

font_title = ImageFont.truetype(font_path, title_size)
font_shlok = ImageFont.truetype(font_path, shlok_size)
font_meaning = ImageFont.truetype(font_path, meaning_size)

# Colors matching the design (Rich Dark Chestnut Brown & Deep Maroon for clarity on parchment)
COLOR_TITLE = (50, 15, 5)
COLOR_SHLOK = (160, 50, 30)
COLOR_TEXT = (60, 30, 15)

for index, row in df.iterrows():
    sr_no = str(row[sr_col]).strip()
    shlok_text = str(row[shlok_col]).strip()
    meaning_text = str(row[meaning_col]).strip()

    # Copy fresh base image
    card = base_template.copy()
    draw = ImageDraw.Draw(card)

    y = text_start_y

    # 1. Heading: 'श्लोक X चा भावार्थ:'
    title_text = f"श्लोक {sr_no} चा भावार्थ:" if sr_no.isdigit() else f"{sr_no} भावार्थ:"
    draw_bold_text(draw, (margin_x, y), title_text, font=font_title, fill=COLOR_TITLE)
    y += int(title_size * 1.5)

    # 2. Sanskrit Shloka (if present)
    if shlok_text and shlok_text.lower() != 'nan':
        shlok_lines = wrap_devanagari_text(shlok_text, font_shlok, content_width, draw)
        for line in shlok_lines:
            draw_bold_text(draw, (margin_x, y), line, font=font_shlok, fill=COLOR_SHLOK)
            y += int(shlok_size * 1.35)
        y += int(shlok_size * 0.4)  # Spacer

    # 3. Marathi Meaning Text
    meaning_lines = wrap_devanagari_text(meaning_text, font_meaning, content_width, draw)
    for line in meaning_lines:
        if y + (meaning_size * 1.2) > (H - int(H * 0.04)):  # Prevent bottom overflow (adjusted line height)
            break
        draw_bold_text(draw, (margin_x, y), line, font=font_meaning, fill=COLOR_TEXT)
        y += int(meaning_size * 1.2) # Adjusted line height

    # Save Image
    safe_sr = str(sr_no).replace(" ", "_").replace(".", "")
    out_filename = os.path.join(output_dir, f"vishnu_shlok_{safe_sr}.png")
    card.convert("RGB").save(out_filename, "PNG", quality=95)

print(f"\nAll {len(df)} images successfully created in '{output_dir}'.")

# 6. Compress and Download Archive
zip_name = "Vishnu_Sahasranamam_Parchment_Cards.zip"
shutil.make_archive("Vishnu_Sahasranamam_Parchment_Cards", 'zip', output_dir)
print(f"Archive generated: {zip_name}. Initiating download...")
files.download(zip_name)

Step 1: Upload the Vishnu Template Image (e.g. vishnu.jpg / vishnu.png):


Saving revimage.jfif to revimage.jfif

Step 2: Upload your Vishnu Sahasranamam CSV file:


Saving vishnu_sahasranamam_full_marathi.csv to vishnu_sahasranamam_full_marathi (5).csv
Loaded 137 rows successfully.

All 137 images successfully created in 'vishnu_generated_cards'.
Archive generated: Vishnu_Sahasranamam_Parchment_Cards.zip. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>